# Data Processing and Merging

In [8]:
import pandas as pd
import numpy as np
import re

In [9]:
DATA_PATH = r"D:\墨大sml作业\data"

ratings_path = DATA_PATH + r"\rating.csv"
movies_path = DATA_PATH + r"\movie.csv"

ratings = pd.read_csv(ratings_path)
movies = pd.read_csv(movies_path)

print(ratings.shape)
print(movies.shape)

(20000263, 4)
(27278, 3)


In [10]:
# 时间处理
ratings["timestamp"] = pd.to_datetime(ratings["timestamp"])

# 二分类标签：rating >= 4 为 like
ratings["label"] = (ratings["rating"] >= 4).astype(int)

# 全局平均分
global_mean = ratings["rating"].mean()
global_mean

np.float64(3.5255285642993797)

In [11]:
# 用户侧 Feature A

user_stats = ratings.groupby("userId").agg(
    user_avg_rating=("rating", "mean"),
    user_rating_count=("rating", "count"),
    user_rating_std=("rating", "std"),
    user_like_count=("label", "sum"),
    user_first_rating_time=("timestamp", "min"),
    user_last_rating_time=("timestamp", "max")
).reset_index()

user_stats["user_like_ratio"] = (
    user_stats["user_like_count"] / user_stats["user_rating_count"]
)

user_stats["user_rating_timespan"] = (
    user_stats["user_last_rating_time"] - user_stats["user_first_rating_time"]
).dt.days

user_stats["user_avg_gap_days"] = user_stats["user_rating_timespan"] / (
    user_stats["user_rating_count"] - 1
)

# 如果用户只评分一次，gap 无法计算，填 0
user_stats["user_avg_gap_days"] = user_stats["user_avg_gap_days"].replace(
    [np.inf, -np.inf], np.nan
).fillna(0)

# std 如果只有一个评分会是 NaN，填 0
user_stats["user_rating_std"] = user_stats["user_rating_std"].fillna(0)

user_stats = user_stats.drop(
    columns=["user_first_rating_time", "user_last_rating_time"]
)

user_stats.head()

,userId,user_avg_rating,user_rating_count,user_rating_std,user_like_count,user_like_ratio,user_rating_timespan,user_avg_gap_days
0,1,3.742857,175,0.382284,88,0.502857,204,1.172414
1,2,4.000000,61,1.064581,43,0.704918,0,0.000000
2,3,4.122995,187,0.910427,145,0.775401,3,0.016129
3,4,3.571429,28,0.790151,16,0.571429,0,0.000000
4,5,4.272727,66,0.969464,50,0.757576,1,0.015385


In [12]:
# 电影侧 Feature A

item_stats = ratings.groupby("movieId").agg(
    item_avg_rating=("rating", "mean"),
    item_rating_count=("rating", "count"),
    item_rating_std=("rating", "std"),
    item_like_count=("label", "sum")
).reset_index()

item_stats["item_like_ratio"] = (
    item_stats["item_like_count"] / item_stats["item_rating_count"]
)

item_stats["item_rating_std"] = item_stats["item_rating_std"].fillna(0)

item_stats.head()

,movieId,item_avg_rating,item_rating_count,item_rating_std,item_like_count,item_like_ratio
0,1,3.921240,49695,0.889012,33294,0.669967
1,2,3.211977,22243,0.951150,7272,0.326934
2,3,3.151040,12735,1.006642,4015,0.315273
3,4,2.861393,2756,1.095702,694,0.251814
4,5,3.064592,12161,0.982140,3288,0.270373


In [13]:
# 从 title 里提取电影发行年份
# 例如 Toy Story (1995) -> 1995

def extract_year(title):
    match = re.search(r"\((\d{4})\)", str(title))
    if match:
        return int(match.group(1))
    return np.nan

movies["movie_year"] = movies["title"].apply(extract_year)

movies[["movieId", "title", "movie_year"]].head()

,movieId,title,movie_year
0,1,Toy Story (1995),1995.0
1,2,Jumanji (1995),1995.0
2,3,Grumpier Old Men (1995),1995.0
3,4,Waiting to Exhale (1995),1995.0
4,5,Father of the Bride Part II (1995),1995.0


In [14]:
# 合并所有 Feature A

df = ratings.merge(user_stats, on="userId", how="left")
df = df.merge(item_stats, on="movieId", how="left")
df = df.merge(movies[["movieId", "movie_year"]], on="movieId", how="left")

# 全局平均分
df["global_mean"] = global_mean

# 评分年份
df["rating_year"] = df["timestamp"].dt.year

# movie_age_at_rating = 评分年份 - 电影发行年份
df["movie_age_at_rating"] = df["rating_year"] - df["movie_year"]

# 如果电影年份缺失，填 0
df["movie_age_at_rating"] = df["movie_age_at_rating"].fillna(0)

df.head()

,userId,movieId,rating,timestamp,label,user_avg_rating,user_rating_count,user_rating_std,user_like_count,user_like_ratio,...,user_avg_gap_days,item_avg_rating,item_rating_count,item_rating_std,item_like_count,item_like_ratio,movie_year,global_mean,rating_year,movie_age_at_rating
0,1,2,3.5,2005-04-02 23:53:47,0,3.742857,175,0.382284,88,0.502857,...,1.172414,3.211977,22243,0.951150,7272,0.326934,1995.0,3.525529,2005,10.0
1,1,29,3.5,2005-04-02 23:31:16,0,3.742857,175,0.382284,88,0.502857,...,1.172414,3.952230,8520,0.975643,5961,0.699648,1995.0,3.525529,2005,10.0
2,1,32,3.5,2005-04-02 23:33:39,0,3.742857,175,0.382284,88,0.502857,...,1.172414,3.898055,44980,0.867135,30091,0.668986,1995.0,3.525529,2005,10.0
3,1,47,3.5,2005-04-02 23:32:07,0,3.742857,175,0.382284,88,0.502857,...,1.172414,4.053493,43249,0.870280,32145,0.743254,1995.0,3.525529,2005,10.0
4,1,50,3.5,2005-04-02 23:29:40,0,3.742857,175,0.382284,88,0.502857,...,1.172414,4.334372,47006,0.756783,39678,0.844105,1995.0,3.525529,2005,10.0


In [15]:
# 最终 Feature A

feature_A = df[
    [
        "userId",
        "movieId",

        # 用户侧 7 个
        "user_avg_rating",
        "user_rating_count",
        "user_rating_std",
        "user_like_count",
        "user_like_ratio",
        "user_rating_timespan",
        "user_avg_gap_days",

        # 电影侧 5 个
        "item_avg_rating",
        "item_rating_count",
        "item_rating_std",
        "item_like_count",
        "item_like_ratio",

        # 全局 + 元数据 2 个
        "global_mean",
        "movie_age_at_rating",

        # label
        "label"
    ]
]

feature_A.head()

,userId,movieId,user_avg_rating,user_rating_count,user_rating_std,user_like_count,user_like_ratio,user_rating_timespan,user_avg_gap_days,item_avg_rating,item_rating_count,item_rating_std,item_like_count,item_like_ratio,global_mean,movie_age_at_rating,label
0,1,2,3.742857,175,0.382284,88,0.502857,204,1.172414,3.211977,22243,0.951150,7272,0.326934,3.525529,10.0,0
1,1,29,3.742857,175,0.382284,88,0.502857,204,1.172414,3.952230,8520,0.975643,5961,0.699648,3.525529,10.0,0
2,1,32,3.742857,175,0.382284,88,0.502857,204,1.172414,3.898055,44980,0.867135,30091,0.668986,3.525529,10.0,0
3,1,47,3.742857,175,0.382284,88,0.502857,204,1.172414,4.053493,43249,0.870280,32145,0.743254,3.525529,10.0,0
4,1,50,3.742857,175,0.382284,88,0.502857,204,1.172414,4.334372,47006,0.756783,39678,0.844105,3.525529,10.0,0


In [16]:
print("Feature A shape:", feature_A.shape)
print("Number of Feature A columns excluding IDs and label:", feature_A.shape[1] - 3)

Feature A shape: (20000263, 17)
Number of Feature A columns excluding IDs and label: 14


In [19]:
OUTPUT_PATH = r"D:\墨大sml作业\Feature1"

output_path = OUTPUT_PATH + r"\feature_A.csv"

feature_A.to_csv(output_path, index=False, encoding="utf-8-sig")

print("Saved to:", output_path)

Saved to: D:\墨大sml作业\Feature1\feature_A.csv


# Preview Data Format

In [20]:
# 保存前50条样例数据

example_path = OUTPUT_PATH + r"\featureA_example2.csv"

feature_A.head(200).to_csv(
    example_path,
    index=False,
    encoding="utf-8-sig"
)

print("Example file saved to:")
print(example_path)

Example file saved to:
D:\墨大sml作业\Feature1\featureA_example2.csv


# Train / test split

In [ ]:
!pip install scikit-learn

Defaulting to user installation because normal site-packages is not writeable


You should consider upgrading via the 'D:\Program Files\Python39\python.exe -m pip install --upgrade pip' command.


In [21]:
from sklearn.model_selection import train_test_split

In [22]:
# 划分训练集和测试集
train_df, test_df = train_test_split(
    feature_A,
    test_size=0.2,
    random_state=42,
    stratify=feature_A["label"]  # 保持 like/dislike 比例一致
)

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)

Train shape: (16000210, 17)
Test shape: (4000053, 17)


In [23]:
train_path = OUTPUT_PATH + r"\feature_A_train.csv"
test_path = OUTPUT_PATH + r"\feature_A_test.csv"

train_df.to_csv(
    train_path,
    index=False,
    encoding="utf-8-sig"
)

test_df.to_csv(
    test_path,
    index=False,
    encoding="utf-8-sig"
)

print("Saved:")
print(train_path)
print(test_path)

Saved:
D:\墨大sml作业\Feature1\feature_A_train.csv
D:\墨大sml作业\Feature1\feature_A_test.csv


# LR